# 03 — Modeling
Train SARIMA, Prophet, XGBoost, LSTM, and Ensemble.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.data_loader import load_processed, train_val_test_split
from src.features import build_feature_matrix, get_feature_columns
from src.models import (SARIMAForecaster, ProphetForecaster,
    XGBoostForecaster, LSTMForecaster, EnsembleForecaster)
%matplotlib inline

In [ ]:
df       = load_processed()
features = build_feature_matrix(df)
train, val, test = train_val_test_split(features)

TARGET   = 'load_mw'
FEAT_COLS = get_feature_columns(features)

X_train, y_train = train[FEAT_COLS], train[TARGET]
X_val,   y_val   = val[FEAT_COLS],   val[TARGET]
X_test,  y_test  = test[FEAT_COLS],  test[TARGET]

## 1 · SARIMA

In [ ]:
sarima = SARIMAForecaster(order=(1,1,1), seasonal_order=(1,1,1,24))
# Train on last 3 months of training data for speed
sarima.fit(train[TARGET]['2015-10-01':])
sarima.save()
print('SARIMA fitted and saved.')

## 2 · Prophet

In [ ]:
prophet = ProphetForecaster(changepoint_prior_scale=0.05)
prophet.fit(train[TARGET])
prophet.save()
print('Prophet fitted and saved.')

## 3 · XGBoost

In [ ]:
xgb = XGBoostForecaster(n_estimators=500, learning_rate=0.05)
xgb.fit(X_train, y_train, X_val=X_val, y_val=y_val)
xgb.save()
print('XGBoost fitted and saved.')

# Feature importance preview
imp = xgb.feature_importance.head(15)
imp.sort_values().plot.barh(figsize=(8,6), color='#2563EB', alpha=0.8)
plt.title('Top 15 XGBoost feature importances'); plt.tight_layout()

## 4 · LSTM

In [ ]:
lstm = LSTMForecaster(lookback=168, horizon=24, units=64)
lstm.fit(train[TARGET], val=val[TARGET], epochs=30, batch_size=64)
lstm.save()
print('LSTM fitted and saved.')

## 5 · Ensemble

In [ ]:
ensemble = EnsembleForecaster({'prophet': prophet, 'xgboost': xgb})
ensemble.fit(val[TARGET], horizon=24, X_val=X_val)
ensemble.save()
print(f'Ensemble weights: {dict(zip(ensemble.forecasters, ensemble.weights_))}')

## Quick 24-hour forecast preview

In [ ]:
fig, ax = plt.subplots(figsize=(14,4))
n = 48  # 2 days
actual_slice = test[TARGET].iloc[:n]
xgb_pred     = xgb.predict(X_test.iloc[:n])
ax.plot(actual_slice.values, color='black', lw=1.5, label='Actual')
ax.plot(xgb_pred,            color='#2563EB', lw=1.2, label='XGBoost', alpha=0.85)
ax.set_title('XGBoost forecast vs actual (first 48h of test set)')
ax.legend(); plt.tight_layout()